In [11]:
import torch
device = "cuda:0" if torch.cuda.is_available() else "cpu"


# Data

In [2]:
import pandas as pd
#dataset
dataset = pd.read_csv('../data/processed/df_processed/final_sample_w_desc_cleaned.csv')

In [13]:
dataset = dataset[['whisper_cc','true_label']]
dataset.columns = ['text','label']

In [14]:
from datasets import Dataset

In [4]:
dataset['true_label'].value_counts()

true_label
1    887
0    779
Name: count, dtype: int64

## Preprocess

In [15]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('roberta-base')
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)



In [16]:
from transformers import DataCollatorWithPadding

# Evaluate

In [17]:
import evaluate

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")
precision = evaluate.load("precision")
recall = evaluate.load("recall")

In [18]:
import numpy as np
from sklearn.metrics import classification_report

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    f1_r = f1.compute(predictions=predictions, references=labels,average='weighted')
    precision_r = precision.compute(predictions=predictions, references=labels,average='weighted')
    recall_r = recall.compute(predictions=predictions, references=labels,average='weighted')
    print(classification_report(labels, predictions))
    return {"f1":f1_r,"precision": precision_r, "recall": recall_r}

# Labels

In [19]:
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

# Model

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# model = AutoModelForSequenceClassification.from_pretrained(
#     "distilbert/distilbert-base-uncased", num_labels=2, id2label=id2label, label2id=label2id
# ).to(device)
model = AutoModelForSequenceClassification.from_pretrained(
"roberta-base", num_labels=2, id2label=id2label, label2id=label2id,force_download = True 
).to(device)

# Training

In [ ]:
from sklearn.model_selection import KFold
kf = KFold(n_splits=5, shuffle=True, random_state=0)

dataset_bert = Dataset.from_pandas(dataset)
dataset_bert = dataset_bert.class_encode_column("label")
tokenized_datasets = dataset_bert.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

for i in range(0,4,1):
    tokenized_dataset_tmp = tokenized_datasets.train_test_split(0.25,stratify_by_column='label',shuffle=True)
    
    training_args = TrainingArguments(
        output_dir="../models/new_baseline",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=2,
        weight_decay=0.01,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        push_to_hub=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset_tmp["train"],
        eval_dataset=tokenized_dataset_tmp["test"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    #trainer.evaluate()